# Data Analysis Project - Python Pandas
## Look at the data -

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

sns.set_theme(style='whitegrid')

flights_raw = pd.read_csv('data/flights.csv')
airports_raw = pd.read_csv('data/airports.csv')

flights = flights_raw.copy()
airports = airports_raw.copy()

print(flights.shape,
      airports.shape)

In [ ]:
print(flights.info(),
      flights.columns,
      flights.value_counts(),
      flights.duplicated().sum(),
      flights.dtypes,
      sep='\n\n')

print(airports.info(),
      airports.columns,
      airports.value_counts(),
      airports.duplicated().sum(),
      airports['iata'].duplicated().sum(),
      airports.dtypes,
      sep='\n\n')

for c in flights.columns:
    print('null', c, flights[c].isna().sum(),
          '; min', c, flights[c].min(),
          '; max', c, flights[c].max())

for c in airports.columns:
    print('null', c, airports[c].isna().sum(),
          '; min', c, airports[c].min(),
          '; max', c, airports[c].max())

## flights.csv (20000,5)
```
 #   Column       Dtype         
---  ------       -----         
 0   date         datetime64[us]
 1   delay        int64         
 2   distance     int64         
 3   origin       str           
 4   destination  str      
```
## We will use all of these columns.
- **issue:** date in str format. **action:** parse dates and check that no NaT.
- **issue:** unknown metrics for delay and distance. **action:** investigate documentation.
- **observation:** there are no duplicate rows.
- **observation:** there are no null values.
## airports.csv (3376, 7)
```
 #   Column     Dtype  
---  ------     -----  
 0   iata       str    
 1   name       str    
 2   city       str    
 3   state      str    
 4   country    str    
 5   latitude   float64
 6   longitude  float64
```
## We will use all of these columsn.
- **issue:** null values city: 12, state: 12. **action:** will investigate and decide if data can be extrapolated or leave.
- **issue:** found a few misspellings. **action:** determine if deliberate abbreviations or fix.
- **observation:** no issues with formats.
- **observation:** iata column data seems correct and can be merged.
- **observation:** there are no duplicate rows.
- **observation:** no duplicate iata codes.
- **observation:** (lat,long) has high precision. keep.

## Data documentation -
The source of both datasets is Vega Datasets (https://vega.github.io/vega-datasets/). The `airports.csv CSV 3,376 rows 7 fields 210 KB` can be found in it's exact form as part of their open source offerings. The `flights.csv` seems to be a converted version of their `flights_20k.json JSON 20,000 rows 5 fields 1.8 MB` dataset. Unfortunately the source documentation does not clarify metrics for **delay and distance**. These will have to be assumed **minutes and miles** respectively.

**airports.csv description:**
Airports in the United States and its territories, including major commercial, regional, and municipal airports. Contains information about each airport's location (latitude/longitude coordinates), identification codes, name, city, state, and country. While the exact generation source of this file is unknown, this data is consistent with files provided on a monthly frequency by the FAA's National Airspace System Resource.

**flights.csv description:**
Flight delay statistics (20,000 rows) from U.S. Bureau of Transportation Statistics. Collected under regulatory reporting requirements (14 CFR Part 234), which mandate that qualifying airlines report on-time performance data to BTS.

## Source of data -
- airports.csv - https://vega.github.io/vega-datasets/datasets/airports/    
Federal Aviation Administration (listed as source by Vega Datasets)
- flights.csv - https://vega.github.io/vega-datasets/datasets/flights_20k/ (in JSON)    
U.S. Bureau of Transportation Statistics (listed as source by Vega Datasets)

## Clean the data -

In [ ]:
# check integrity iata fields
for c in ['origin', 'destination']:
    print('min len', c, flights[c].str.len().min())
    print('max len', c, flights[c].str.len().max())
    print('A-Z 0-9', c, flights[c].str.fullmatch(r"[A-Z0-9]+").sum())

print('min len', airports['iata'].str.len().min())
print('max len', airports['iata'].str.len().max()) 
# we find iata with max len of 4
print('A-Z 0-9', airports['iata'].str.fullmatch(r"[A-Z0-9]+").sum())

# narrow down len 4 airports['iata']
print('len 4', airports.loc[airports['iata'].str.len() == 4, 'iata'].head())
print('count len 4', airports.loc[airports['iata'].str.len() == 4, 'iata'].count()) 
# 42 iata with 4 characters.

# check state and country
print('country', airports['country'].value_counts())
# 4 outlier country with only 1 entry. rest USA.
# Thailand 1, Palau 1, N Mariana Islands 1, Federated States of Micronesia 1
print('unique state', airports['state'].nunique())
# 56 unique states. too many, maybe grouped with outlier countries.
print('state', airports['state'].value_counts().tolist())
# distrubution seems ok.

# checking flight distance outlier
print('min_dist', flights['distance'].min(), 
      'max_dist', flights['distance'].max())
# longest operational flight within US is Boston - Honolulu 5095mi. 4475 less so ok.

In [ ]:
# check 4 outlier contries
country_usa = airports['country'].isin(['USA'])
print(airports.loc[~country_usa, ['iata', 'country', 'state']])
# all outlier countries have null state. 4/4

#check null city, state
null_city = airports['city'].isna()
null_state = airports['state'].isna()
print('exact match', null_city.equals(null_state))
# 12 missing values are 12 rows where both city and state are null

print(airports.loc[null_city, ['iata', 'city', 'state', 'country']])
print(airports.loc[null_state, ['iata', 'city', 'state', 'country']])
# we see that the null cities and states are overlapped
# and include the 4 outlier countries.

# see if outliers are used in flights
null_iata = list(airports.loc[airports['city'].isna(),['iata']])
print('null values used in flights', 
      flights[flights['origin'].isin(null_iata)]['date'].count(),
      flights[flights['destination'].isin(null_iata)]['date'].count())
# none of the null value city, state, outlier countries are used in flights
# we will disregard this faulty data

# check state formatting
state_AZ_char = airports['state'].str.fullmatch(r"[A-Z]+")
print('A-Z state', state_AZ_char.sum())
# 3364 correct format. 3376 - 12 null

# check city formatting
city_AZaz_char = airports['city'].str.fullmatch(r"[A-Za-z\ ]+")
print('A-Z a-z city', city_AZaz_char.sum())
# 3261. 103 short of total - null.
print(airports[~city_AZaz_char]['city'])
city_AZaz_extra = airports['city'].str.fullmatch(r"[A-Za-z\ \/]+")
print('city format with special', city_AZaz_extra.sum())
# 3306 / char also used
print(airports[~city_AZaz_extra]['city'])
# - . ( ) ' , used
city_AZaz_more_extra = airports['city'].str.fullmatch(r"[A-Za-z\ \/\-\.\(\)\'\,]+")
print('city format full special', city_AZaz_more_extra.sum())
# 3364 correct format with these acceptable special characters. total -12 null.

# check name formatting
name_with_special = airports['name'].str.fullmatch(r"[A-Za-z\ \/\-\.\(\)\'\,\"\:\&]+")
print('name format', name_with_special.sum())
# 3367 using all acceptable special characters
print(airports[~name_with_special]['name'])
# the 8 outliers contain digits.
# 2 with valid use of digits. 6 ending with 2 or Nr 2.

# check if 2 / Nr 2 are used in flights
ok_names = airports['name'].str.fullmatch(r"[A-Za-z\ \/\-\.\(\)\'\,\"\:\&]+")
name_2_iata = airports.loc[~ok_names,'iata']
print('2 / Nr2 names used in flights', 
      flights[flights['origin'].isin(name_2_iata)]['date'].count(),
      flights[flights['destination'].isin(name_2_iata)]['date'].count())
# not used in flights. disregard

# check for double spaces and valid leading character
for c in ['name', 'city', 'state', 'country']:
    double_space = airports[c].str.contains('  ', regex=False, na=False)
    az_start = airports[c].str.match(r'^[A-Z]')
    print('leading char not AZ', c, airports[~az_start]['name'].count())
    print('double space in', c, airports[double_space]['name'].count())
# name contains 12 entries containing double spaces
# all fields start with capital letter. excluding the NaN


- **issue:** 4 outlier countries. **action:** investigate if they are used in flights.
- **issue:** 42 outlier 4 character iata codes. **action:** investigate if 4 character are valid.
- **issue:** 56 unique states. **action:** investigate why there are too many.
- **observation:** the 12 null values in city / state are overlapping.
- **observation:** the 4 outlier countries are part of the 12 null city / state.
- **issue:** null values and outliers. **action:** investigate if they are used in flights.
- **observation:** all city, state and country using acceptable characters and formatting.
- **issue:** 8 invalid `name`. 2 valid use of digits. 6 potential duplicates using '2' or 'Nr 2'. **action:** investigate if true duplicates or just valid secondary location.
- **issue:** 12 entries in name with double white space character. **action:** replace with single white space character.
- **observation:** all string fields have valid leading character of capital letter.
- **observation:** none of the null city, state and outlier countries are used in flights. Keep / Disregard.
- **observation:** names ending in '2' or 'Nr 2' not present in flights. Keep / Disregard.

### What to merge from airports
all fields we merge will be merged under both sets origin_ and destination_ on iata for those fields.
- iata - obsolete after merge
- name - probably not needed but we'll keep it
- city - keep
- state - keep
- country - all flights are US flights. not necessary.
- latitude - keep and shorten to lat
- longitude - keep and shorten to long

## Merging the Data -

In [ ]:
def merge_set(flights, airports, node: str):
    merge = airports[['iata', 'name', 'city', 'state', 'latitude', 'longitude']]
    merge = merge.add_prefix(node+'_')

    return (flights.merge(
        merge, left_on=node, right_on=f'{node}_iata',
        how='left', validate='many_to_one')
        .drop(columns=f'{node}_iata'))

def clean_merge(flights, airports):

    # parse dates
    flights['date'] = pd.to_datetime(flights['date'], errors="coerce")
    print('min year', flights['date'].min(), 
          'max year', flights['date'].max())
    print('invalid dates', flights[flights['date'].isna()]['date'].count())

    # clean double spaces
    double_space = airports['name'].str.contains('  ', regex=False, na=False)
    airports['name'].replace(r'\s+', ' ', regex=True)
    print('double space in', airports[double_space]['name'].count())

    # add distance_km
    flights['distance_km'] = (flights['distance'] * 1.609344).round().astype(int)
    print('min_km', flights['distance_km'].min(), 
          'max_km', flights['distance_km'].max())

    # merge airport data sets of origin_ and destination_
    flights = merge_set(flights, airports, 'origin')
    flights = merge_set(flights, airports, 'destination')
    print(flights.shape)

    return flights

clean = clean_merge(flights, airports)

print(clean.loc[clean['destination_city'].isna()])

for c in clean.columns:
    print('null', c, clean[c].isna().sum(),
          '; min', c, clean[c].min(),
          '; max', c, clean[c].max())